## MODEL A ( All Features )

### 1. Import the libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import joblib


### 2. Load the dataset

In [2]:
data = pd.read_csv(r"C:\Users\HP\Documents\PycharmProjects\network-anomaly-detector\Detector\data\processed-data\clean-data.csv", low_memory=False)
print(data.shape)
data.sample(10)

(2540047, 45)


,sport,dsport,proto,state,dur,sbytes,dbytes,sttl,dttl,service,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
364295,27383.0,5190.0,tcp,FIN,0.153236,1244,2574,31,29,unknown,...,0,3,9,3,3,1,1,1,NaN,0
1415621,1043.0,53.0,udp,INT,0.000009,114,0,254,0,dns,...,0,35,35,25,28,25,25,35,Generic,1
2091779,59141.0,143.0,tcp,FIN,0.031176,7816,15020,31,29,unknown,...,0,7,3,2,2,1,1,1,NaN,0
1733407,59656.0,64545.0,tcp,FIN,0.028083,3856,2560,31,29,unknown,...,0,3,3,4,4,1,1,4,NaN,0
2173916,47439.0,53.0,udp,INT,0.000007,264,0,60,0,dns,...,0,42,42,42,42,42,19,42,NaN,0
1161269,47439.0,53.0,udp,INT,0.000005,264,0,60,0,dns,...,0,15,15,8,8,8,8,15,NaN,0
2230519,60383.0,111.0,udp,CON,0.251181,568,304,31,29,unknown,...,0,5,6,4,6,1,1,4,NaN,0
1993034,16833.0,22.0,tcp,FIN,0.275945,9400,12298,31,29,ssh,...,0,1,1,2,2,1,1,1,NaN,0
387306,36541.0,6881.0,tcp,FIN,5.477864,12986,531132,31,29,unknown,...,0,7,6,3,1,1,1,1,NaN,0
1604773,8770.0,520.0,udp,INT,0.000010,1064,0,254,0,unknown,...,0,34,34,10,10,10,1,34,NaN,0


### 3. Train-Test-Split

In [4]:
X_train,X_test,y_train,y_test = train_test_split(
    data.drop(columns=["Label","attack_cat"]),
    data["Label"],
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(2032037, 43)
(508010, 43)
(2032037,)
(508010,)


attack_cat was dropped because it will be used in the next model, not needed here

### 4. Separate the diffent type of features

In [6]:
numeric_cols = X_train.select_dtypes(
    include=[np.number]
).columns
categorical_cols = X_train.select_dtypes(
    exclude=[np.number]
).columns

print("Numeric Features :\n",numeric_cols)
print("\nCategorical Features:\n",categorical_cols)

Numeric Features :
 Index(['sport', 'dsport', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'Sload',
       'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime',
       'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm'],
      dtype='str')

Categorical Features:
 Index(['proto', 'state', 'service'], dtype='str')


Making custom transformer for grouping low frequency categories

In [7]:
from sklearn.base import BaseEstimator, TransformerMixin

class RareCategories(BaseEstimator, TransformerMixin):
    def __init__(self, top_n=7):
        self.top_n = top_n
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            self.top_categories_[col] = (
                X[col].value_counts()
                .head(self.top_n)
                .index
                .tolist()
            )
        return self

    def transform(self, X):
        X = X.copy()

        for col in X.columns:
            X[col] = X[col].where(
                X[col].isin(self.top_categories_[col]),
                "other"
            )

        return X

### 5. Creating pipelines for Feature Engineering

In [ ]:
num_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most frequent')),
    ('Scaler', StandardScaler()),
])
cat_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent').set_output(transform = 'pandas')),
    ('Grouping', RareCategories(top_n=7)),
    ('Encoder', OneHotEncoder(handle_unknown='ignore'))
])

### 6. Using Column Transformer

In [ ]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numeric_cols),
    ("cat", cat_pipeline, categorical_cols),
])

### 7. Creating pipelines for models

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

### 8. Model training

In [ ]:
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

lr_pred = lr_pipeline.predict(X_test)
rf_pred = rf_pipeline.predict(X_test)

### 9. Model Evaluation

## MODEL B ( High Correlation Features )